# Train / Test Split

This notebook runs once. It produces:
- `X_train.npy`, `X_test.npy`, `y_train.npy`, `y_test.npy`
- `scaler.pkl` — fitted on training data only
- `feat_cols.npy` — ordered list of feature column names

Every subsequent modelling notebook loads these files directly.  
**The test set is locked after this notebook runs. Do not use it until final evaluation.**

## Step 1 — Load data and define feature columns

**Why explicit column selection matters:**  
features_model.csv has 330 columns — label + 311 hist + 13 bins + 5 scalars.  
We explicitly name the 18 columns going into the model rather than using positional indexing.  
This makes it impossible to accidentally include the wrong features or change the column order silently.

In [ ]:
import numpy as np
import pandas as pd
import sys, pathlib

ROOT = pathlib.Path().resolve().parent
sys.path.insert(0, str(ROOT))

df = pd.read_csv(ROOT / 'data/processed/features_model.csv')

# 13 bin features + 5 scalars = 18 features total
bin_cols    = [c for c in df.columns if c.startswith('bin_')]
scalar_cols = ['short_long_ratio', 'mean_length', 'median_length',
               'mono_peak_height', 'peak_to_flank']
feat_cols   = bin_cols + scalar_cols

print(f'Total features : {len(feat_cols)}')
print(f'  Bin columns  : {len(bin_cols)}')
print(f'  Scalar cols  : {len(scalar_cols)}')
print()
print('Feature list:')
for c in feat_cols:
    print(f'  {c}')

---
## Test Set Lock

**The test set is now locked.**

- `X_test.npy` and `y_test.npy` are not loaded by any model training notebook
- No hyperparameter tuning, threshold selection, or feature decisions are made using test data
- The test set is used exactly once — in the final evaluation notebook — to report AUC, sensitivity, and specificity

**What goes into training notebooks:**
- `X_train.npy` — 148 scaled samples
- `y_train.npy` — 148 labels
- 5-fold stratified cross-validation on the training set for all model selection decisions

Violating the lock invalidates the reported AUC. A model tuned on test data is not a model — it is a memorisation of the test set.

In [ ]:
import pickle

SPLIT_DIR = ROOT / 'data' / 'processed' / 'split'
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

np.save(SPLIT_DIR / 'X_train.npy',   X_train_sc)
np.save(SPLIT_DIR / 'X_test.npy',    X_test_sc)
np.save(SPLIT_DIR / 'y_train.npy',   y_train)
np.save(SPLIT_DIR / 'y_test.npy',    y_test)
np.save(SPLIT_DIR / 'feat_cols.npy', np.array(feat_cols))

with open(SPLIT_DIR / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Saved to:', SPLIT_DIR)
print()
for p in sorted(SPLIT_DIR.iterdir()):
    print(f'  {p.name:<20}  {p.stat().st_size:>8,} bytes')

## Step 5 — Save all artefacts

Everything produced by this notebook is saved to `data/processed/split/`.  
Downstream notebooks load from there — they never re-split or re-fit the scaler.

Files saved:
- `X_train.npy`, `X_test.npy` — scaled feature arrays
- `y_train.npy`, `y_test.npy` — binary labels
- `feat_cols.npy` — ordered feature names (needed for SHAP and interpretability)
- `scaler.pkl` — fitted scaler (needed to transform new samples at inference time)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit on train, transform train
X_test_sc  = scaler.transform(X_test)        # transform only — no fit

print('Scaler fitted on training data only.')
print(f'X_train_sc — mean: {X_train_sc.mean():.4f}  std: {X_train_sc.std():.4f}')
print(f'X_test_sc  — mean: {X_test_sc.mean():.4f}   std: {X_test_sc.std():.4f}')
print()
print('Note: train mean≈0, std≈1 by construction.')
print('Test mean/std will differ slightly — that is correct and expected.')

## Step 4 — Fit scaler on training data only

**The data leakage rule:**  
`scaler.fit()` computes mean and std from the data it sees.  
If you call `fit()` on the full dataset, test samples contribute to those statistics — the model has indirectly seen test data before evaluation. That's leakage.  

Correct sequence:
1. `fit(X_train)` — learn statistics from training data only  
2. `transform(X_train)` — normalise training data  
3. `transform(X_test)` — normalise test data using **training** statistics  

Never call `fit()` or `fit_transform()` on test data.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print(f'Train : {X_train.shape[0]} samples  '
      f'(healthy={( y_train==0).sum()}  cancer={(y_train==1).sum()})')
print(f'Test  : {X_test.shape[0]} samples  '
      f'(healthy={(y_test==0).sum()}  cancer={(y_test==1).sum()})')
print()
train_ratio = (y_train==0).sum() / (y_train==1).sum()
test_ratio  = (y_test==0).sum()  / (y_test==1).sum()
print(f'Healthy:Cancer ratio — train: {train_ratio:.2f}  test: {test_ratio:.2f}  (should match ~1.78)')

## Step 3 — Stratified 80/20 split

**Why stratified:**  
A plain random split could put almost all cancer samples in training by chance.  
Stratified guarantees the 1.78:1 healthy/cancer ratio is preserved in both train and test.

**Why random_state=42, never change it:**  
This locks the exact partition. Every model notebook trains on the same 149 samples and evaluates on the same 37. If you change the seed, you're comparing models trained on different data — results become meaningless.

In [ ]:
X = df[feat_cols].astype(float).values
y = (df['label'] == 'cancer').astype(int).values

print(f'X shape : {X.shape}  (samples × features)')
print(f'y shape : {y.shape}')
print(f'y=0 (healthy) : {(y==0).sum()}')
print(f'y=1 (cancer)  : {(y==1).sum()}')

## Step 2 — Define the target variable

**Why cancer=1:**  
The model's output will be a probability score between 0 and 1.  
Setting cancer=1 means that score is directly interpretable as *"probability this sample is cancer"* — which is what a clinician wants. If we flipped it, a high score would mean healthy, which is confusing.

`X` = the 18 feature values for all 186 samples (shape: 186 × 18)  
`y` = the binary label for all 186 samples (shape: 186,)